# Notebook 01 — Train / Val / Test Split

**Mục đích:** Đọc raw dataset, kiểm tra sơ bộ, sau đó chia thành
ba tập Train / Val / Test bằng Stratified Split tự cài đặt, cuối
cùng lưu ba file CSV vào `data/processed/`.

**Lưu ý anti-leakage:** Notebook này **KHÔNG** thực hiện bất kỳ
bước làm sạch hay chuẩn hoá nào. Ba file được lưu vẫn còn nguyên
missing/duplicate/outlier — đúng nghĩa "raw after split". Mọi bước
preprocessing đều được thực hiện ở `02_eda_preprocessing.ipynb`,
và tham số preprocessing chỉ được fit trên `train_raw.csv`.

## 0. Setup

In [2]:
import sys
import pathlib

# Notebook nằm trong notebooks/, nên gốc project là thư mục cha (.parent)
PROJECT_ROOT = pathlib.Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.preprocessing import load_raw_data, split_data

# Đường dẫn dữ liệu (tuyệt đối, dựa trên PROJECT_ROOT)
RAW_DATA_PATH   = str(PROJECT_ROOT / 'data' / 'raw' / 'Wine.csv')
PROCESSED_DIR   = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'RAW_DATA_PATH : {RAW_DATA_PATH}')
print(f'PROCESSED_DIR : {PROCESSED_DIR.resolve()}')

PROJECT_ROOT  : D:\ĐH_GTVT\Năm 2\HK hè năm 2\Machine Learning\PCA\project for PCA
RAW_DATA_PATH : D:\ĐH_GTVT\Năm 2\HK hè năm 2\Machine Learning\PCA\project for PCA\data\raw\Wine.csv
PROCESSED_DIR : D:\ĐH_GTVT\Năm 2\HK hè năm 2\Machine Learning\PCA\project for PCA\data\processed


---
## Section 1: Đọc Raw Dataset

Đọc file `data/raw/Wine.csv` bằng hàm `load_raw_data()` từ module
`src.preprocessing`. Hàm này chỉ đơn giản đọc CSV và trả về
DataFrame nguyên trạng — không làm sạch, không biến đổi gì.

Mục tiêu ở bước này: hiểu cấu trúc dataset (shape, kiểu dữ liệu,
các cột feature) để chuẩn bị cho bước kiểm tra và split.

In [3]:
# ── Đọc CSV ───────────────────────────────────────────────────────────
df = load_raw_data(RAW_DATA_PATH)

print(f'Shape: {df.shape}  '  # (181, 14)
      f'→ {df.shape[0]} dòng, {df.shape[1]} cột')
print(f'Cột  : {list(df.columns)}')

Shape: (181, 14)  → 181 dòng, 14 cột
Cột  : ['Alcohol', 'Malic_Acid', 'Ash', 'Ash_Alcanity', 'Magnesium', 'Total_Phenols', 'Flavanoids', 'Nonflavanoid_Phenols', 'Proanthocyanins', 'Color_Intensity', 'Hue', 'OD280', 'Proline', 'Customer_Segment']


In [4]:
# ── 5 dòng đầu ────────────────────────────────────────────────────────
df.head()

,Alcohol,Malic_Acid,Ash,Ash_Alcanity,Magnesium,Total_Phenols,Flavanoids,Nonflavanoid_Phenols,Proanthocyanins,Color_Intensity,Hue,OD280,Proline,Customer_Segment
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,1
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,1
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,1
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,1
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,1


In [5]:
# ── Kiểu dữ liệu từng cột ─────────────────────────────────────────────
print('=== df.dtypes ===')
print(df.dtypes)
print(f'\nSố cột kiểu float64 : {(df.dtypes == "float64").sum()}')
print(f'Số cột kiểu int64   : {(df.dtypes == "int64").sum()}')
print(f'Số cột kiểu object  : {(df.dtypes == "object").sum()}')

=== df.dtypes ===
Alcohol                 float64
Malic_Acid              float64
Ash                     float64
Ash_Alcanity            float64
Magnesium               float64
Total_Phenols           float64
Flavanoids              float64
Nonflavanoid_Phenols    float64
Proanthocyanins         float64
Color_Intensity         float64
Hue                     float64
OD280                   float64
Proline                 float64
Customer_Segment          int64
dtype: object

Số cột kiểu float64 : 13
Số cột kiểu int64   : 1
Số cột kiểu object  : 0


---
## Section 2: Kiểm Tra Nhanh Trước Khi Split

Kiểm tra sơ bộ chất lượng dữ liệu raw để biết tổng quan về các
vấn đề cần xử lý sau này:

- **Missing values**: các ô bị thiếu (NaN) theo từng cột
- **Duplicate rows**: dòng bị lặp hoàn toàn
- **Phân bố nhãn** `Customer_Segment`: quan trọng để đánh giá mức độ
  mất cân bằng và xác nhận Stratified Split sau này hoạt động đúng

> **Lưu ý:** Bước này chỉ quan sát, **chưa xử lý** bất kỳ vấn đề nào.
> Toàn bộ việc làm sạch (impute, drop dup, clip outlier) sẽ được thực
> hiện ở `02_eda_preprocessing.ipynb`.

In [6]:
# ── Missing values ────────────────────────────────────────────────────
missing_per_col = df.isnull().sum()
total_missing   = int(missing_per_col.sum())

print(f'Tổng số ô bị missing: {total_missing}')
print()
if total_missing > 0:
    print('Chi tiết theo cột:')
    print(missing_per_col[missing_per_col > 0].to_string())
else:
    print('Không có missing value.')

Tổng số ô bị missing: 5

Chi tiết theo cột:
Ash    3
Hue    2


In [7]:
# ── Duplicate rows ────────────────────────────────────────────────────
n_dup = int(df.duplicated().sum())
print(f'Số dòng duplicate: {n_dup}')
if n_dup > 0:
    print()
    print('Các dòng bị lặp (hiển thị bản sao, không phải dòng gốc):')
    display(df[df.duplicated(keep="first")])

Số dòng duplicate: 3

Các dòng bị lặp (hiển thị bản sao, không phải dòng gốc):


,Alcohol,Malic_Acid,Ash,Ash_Alcanity,Magnesium,Total_Phenols,Flavanoids,Nonflavanoid_Phenols,Proanthocyanins,Color_Intensity,Hue,OD280,Proline,Customer_Segment
178,13.69,3.26,2.54,20.0,107.0,1.83,0.56,0.50,0.80,5.88,0.96,1.82,680.0,3
179,12.42,1.61,2.19,22.5,108.0,2.00,2.09,0.34,1.61,2.06,1.06,2.96,345.0,2
180,13.64,3.10,NaN,15.2,116.0,2.70,3.03,0.17,1.66,5.10,0.96,3.36,845.0,1


In [8]:
# ── Phân bố nhãn Customer_Segment ─────────────────────────────────────
TARGET_COL = 'Customer_Segment'

vc = df[TARGET_COL].value_counts().sort_index()
vc_pct = df[TARGET_COL].value_counts(normalize=True).sort_index() * 100

print(f'Phân bố nhãn "{TARGET_COL}":')
print(f'{"Lớp":>6}  {"Số dòng":>8}  {"Tỉ lệ":>8}')
print('-' * 28)
for cls in vc.index:
    print(f'{cls:>6}  {vc[cls]:>8d}  {vc_pct[cls]:>7.1f}%')
print('-' * 28)
print(f'{"Tổng":>6}  {vc.sum():>8d}  {100.0:>7.1f}%')

Phân bố nhãn "Customer_Segment":
   Lớp   Số dòng     Tỉ lệ
----------------------------
     1        60     33.1%
     2        72     39.8%
     3        49     27.1%
----------------------------
  Tổng       181    100.0%


---
## Section 3: Stratified Train / Val / Test Split

Gọi hàm `split_data()` đã tự cài đặt trong `src/preprocessing.py`
(không dùng sklearn) để chia dataset theo tỉ lệ **70 / 15 / 15**.

### Tại sao Stratified?
Nếu split ngẫu nhiên đơn thuần, một tập có thể nhận ít hoặc nhiều
mẫu của một lớp hơn tập kia. Stratified Split đảm bảo **tỉ lệ nhãn
`Customer_Segment` được bảo toàn** ở cả ba tập, giúp:
- Mô hình học đủ mẫu của mọi lớp.
- Kết quả đánh giá (Val/Test) phản ánh đúng phân bố thực tế.

### Anti-Leakage
Split được thực hiện **trước** mọi bước fit tham số (impute median,
tính IQR, fit scaler). Điều này đảm bảo Val và Test không bị "nhìn"
vào quá trình học tham số, tránh data leakage.

In [9]:
train_df, val_df, test_df = split_data(
    df,
    target_col  = TARGET_COL,
    test_size   = 0.15,
    val_size    = 0.15,
    random_state= 42,
)

[split_data] Tổng: 181 dòng | random_state=42
  Train :  127 dòng  (70.2%)
  Val   :   27 dòng  (14.9%)
  Test  :   27 dòng  (14.9%)
  Train phân bố Customer_Segment: {1: '33.1%', 2: '39.4%', 3: '27.6%'}
  Val phân bố Customer_Segment: {1: '33.3%', 2: '40.7%', 3: '25.9%'}
  Test phân bố Customer_Segment: {1: '33.3%', 2: '40.7%', 3: '25.9%'}


In [10]:
# ── Xác nhận shape ────────────────────────────────────────────────────
total = len(df)
print(f'{'Tập':<8} {'Dòng':>6} {'Tỉ lệ':>8}')
print('-' * 26)
for name, subset in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name:<8} {len(subset):>6d} {len(subset)/total*100:>7.1f}%')
print('-' * 26)
print(f'{'Tổng':<8} {total:>6d} {100.0:>7.1f}%')

Tập        Dòng    Tỉ lệ
--------------------------
Train       127    70.2%
Val          27    14.9%
Test         27    14.9%
--------------------------
Tổng        181   100.0%


In [11]:
# ── Xác nhận phân bố nhãn ở mỗi tập (Stratified đúng?) ───────────────
print(f'Phân bố "{TARGET_COL}" sau Stratified Split:\n')
print(f'{'Lớp':>6}  {'Train':>8}  {'Val':>8}  {'Test':>8}')
print('-' * 38)
for cls in sorted(df[TARGET_COL].unique()):
    t_pct = (train_df[TARGET_COL] == cls).mean() * 100
    v_pct = (val_df[TARGET_COL]   == cls).mean() * 100
    e_pct = (test_df[TARGET_COL]  == cls).mean() * 100
    print(f'{cls:>6}  {t_pct:>7.1f}%  {v_pct:>7.1f}%  {e_pct:>7.1f}%')
print('-' * 38)
print('→ Tỉ lệ nhãn xấp xỉ nhau ở cả 3 tập → Stratified Split thành công.')

Phân bố "Customer_Segment" sau Stratified Split:

   Lớp     Train       Val      Test
--------------------------------------
     1     33.1%     33.3%     33.3%
     2     39.4%     40.7%     40.7%
     3     27.6%     25.9%     25.9%
--------------------------------------
→ Tỉ lệ nhãn xấp xỉ nhau ở cả 3 tập → Stratified Split thành công.


---
## Section 4: Lưu Dữ Liệu Thô Đã Chia

Lưu ba DataFrame vừa split vào `data/processed/` dưới dạng CSV.

> **Quan trọng:** Ba file `train_raw.csv`, `val_raw.csv`,
> `test_raw.csv` **vẫn còn nguyên** missing values, duplicate rows
> và outlier — đây là dữ liệu "raw after split", chưa được làm sạch.
>
> Việc làm sạch (impute, drop dup, clip outlier) và chuẩn hoá sẽ
> được thực hiện ở **`02_eda_preprocessing.ipynb`**, và **toàn bộ
> tham số preprocessing chỉ được fit trên `train_raw.csv`** để
> tránh data leakage sang Val và Test.

In [12]:
# ── Lưu 3 file CSV ────────────────────────────────────────────────────
train_path = PROCESSED_DIR / 'train_raw.csv'
val_path   = PROCESSED_DIR / 'val_raw.csv'
test_path  = PROCESSED_DIR / 'test_raw.csv'

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path,     index=False)
test_df.to_csv(test_path,   index=False)

print('Da luu:')
for p, subset in [(train_path, train_df),
                  (val_path,   val_df),
                  (test_path,  test_df)]:
    print(f'  {str(p):<45}  shape={subset.shape}')

Da luu:
  D:\ĐH_GTVT\Năm 2\HK hè năm 2\Machine Learning\PCA\project for PCA\data\processed\train_raw.csv  shape=(127, 14)
  D:\ĐH_GTVT\Năm 2\HK hè năm 2\Machine Learning\PCA\project for PCA\data\processed\val_raw.csv  shape=(27, 14)
  D:\ĐH_GTVT\Năm 2\HK hè năm 2\Machine Learning\PCA\project for PCA\data\processed\test_raw.csv  shape=(27, 14)


In [13]:
# ── Xác nhận file tồn tại và đọc lại được ────────────────────────────
print('Xac nhan doc lai:')
for p in [train_path, val_path, test_path]:
    tmp = pd.read_csv(p)
    missing_check = tmp.isnull().sum().sum()
    dup_check     = tmp.duplicated().sum()
    print(f'  {p.name:<18}  shape={tmp.shape}  '
          f'missing={missing_check}  dup={dup_check}')

print()
print('Luu y: missing va dup van con la dung - se xu ly o notebook 02.')

Xac nhan doc lai:
  train_raw.csv       shape=(127, 14)  missing=4  dup=1
  val_raw.csv         shape=(27, 14)  missing=0  dup=0
  test_raw.csv        shape=(27, 14)  missing=1  dup=0

Luu y: missing va dup van con la dung - se xu ly o notebook 02.


---
## Kết Luận Notebook 01

| File | Số dòng | Missing | Duplicate |
|------|--------:|--------:|----------:|
| `train_raw.csv` | ~127 | còn | còn |
| `val_raw.csv`   | ~27  | còn | còn |
| `test_raw.csv`  | ~27  | còn | còn |

**Bước tiếp theo:** Mở `02_eda_preprocessing.ipynb` để:
1. EDA chi tiết (phân phối, correlation, outlier visualization)
2. Fit cleaning params **chỉ từ** `train_raw.csv`
3. Clean cả 3 tập (impute → drop dup → clip)
4. Fit scaler **chỉ từ** train_clean
5. Scale cả 3 tập → numpy array sẵn sàng đưa vào PCA